In [1]:
import pandas as pd
import json
from pathlib import Path

# Define the base directory
base_dir = Path('extracted_data')

# List to store all equipment data
all_equipment = []

# Iterate through year folders
for year_folder in sorted(base_dir.glob('*')):
    if year_folder.is_dir():
        # Iterate through JSON files in each year folder
        for json_file in year_folder.glob('*.json'):
            try:
                # Read JSON file
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                # Extract permit information
                permit_date = data.get('permitIssuanceDate', '')
                registration_number = data.get('registrationNumber', '')
                data_center_name = data.get('dataCenterName', '')
                
                # Process each equipment item
                for equipment in data.get('equipmentSummary', []):
                    equipment_row = {
                        'Permit Date': permit_date,
                        'Registration Number': registration_number,
                        'Data Center Name': data_center_name,
                        'Type': equipment.get('type'),
                        'Reference No.': equipment.get('referenceNos'),
                        'Description': equipment.get('description'),
                        'Manufacturer (AI Inferred)': equipment.get('manufacturer'),
                        'Fuel Type (AI Inferred)': equipment.get('fuelType'),
                        'No. of Units': equipment.get('numberOfUnits'),
                        'Electrical Capacity KW Per Unit': equipment.get('electricalCapacity_kW_perUnit'),
                        'Total Electrical Capacity (AI Inferred)': equipment.get('electricalCapacity_kW_total'),
                        'Mechanical Capacity bhp Per Unit': equipment.get('mechanicalCapacity_bhp_perUnit'),
                        'Total Mechanical Capacity (AI Inferred)': equipment.get('mechanicalCapacity_bhp_total'),
                        'Gas Usage': equipment.get('gasUsage_MMBTUhr_perUnit'),
                        'Controls (AI Inferred)': equipment.get('controls'),
                        'Add On Control Tech': equipment.get('addOnControlTechnology'),
                        'Original Permit Date (AI Inferred)': equipment.get('originalPermitDate')
                    }
                    all_equipment.append(equipment_row)
            
            except Exception as e:
                print(f"Error processing {json_file}: {e}")

# Create DataFrame
df = pd.DataFrame(all_equipment)

# Display the DataFrame
print(f"Total equipment records: {len(df)}")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())

# Display first few rows
df.head()

Total equipment records: 819

DataFrame shape: (819, 17)

Column names:
['Permit Date', 'Registration Number', 'Data Center Name', 'Type', 'Reference No.', 'Description', 'Manufacturer (AI Inferred)', 'Fuel Type (AI Inferred)', 'No. of Units', 'Electrical Capacity KW Per Unit', 'Total Electrical Capacity (AI Inferred)', 'Mechanical Capacity bhp Per Unit', 'Total Mechanical Capacity (AI Inferred)', 'Gas Usage', 'Controls (AI Inferred)', 'Add On Control Tech', 'Original Permit Date (AI Inferred)']


,Permit Date,Registration Number,Data Center Name,Type,Reference No.,Description,Manufacturer (AI Inferred),Fuel Type (AI Inferred),No. of Units,Electrical Capacity KW Per Unit,Total Electrical Capacity (AI Inferred),Mechanical Capacity bhp Per Unit,Total Mechanical Capacity (AI Inferred),Gas Usage,Controls (AI Inferred),Add On Control Tech,Original Permit Date (AI Inferred)
0,2000-11-22,73170,"Exodus Communications, Incorporated (DC1)",Constructed,Two Caterpillar 3508B Generators,Caterpillar model 3508B diesel-fired generators,Caterpillar,Diesel,2.0,1000.0,2000.0,NaN,NaN,NaN,null,None,None
1,2003-09-16,73158,Qwest Communications Corporation,Modified,"S2, 1-10",ten Cummins Onan 1500DFLE Diesel Genset,Cummins Onan,Diesel,10.0,1500.0,15000.0,NaN,NaN,NaN,null,None,2001-06-29
2,2003-09-16,73158,Qwest Communications Corporation,Modified,"S1, 1-3",three Caterpillar 3516B Diesel Genset,Caterpillar,Diesel,3.0,2000.0,6000.0,NaN,NaN,NaN,null,None,2001-06-29
3,2005-07-14,73326,Unisys Corporation,Constructed,Reference No. 1,"Caterpillar diesel fired emergency generator, ...",Caterpillar,Diesel,1.0,1250.0,1250.0,1818.0,1818.0,NaN,null,None,None
4,2006-10-19,73200,Verizon Business,Previously Permitted,EGU1 thru EGU3,Three Caterpillar model 3516B diesel engine-dr...,Caterpillar,Diesel,3.0,2000.0,6000.0,2848.0,8544.0,NaN,"Caterpillar's low emission 'B' package, good c...",None,None


In [2]:
# Process files from retries folder with different schema
retries_dir = Path('retries')

# List to store equipment data from retries folder
retries_equipment = []

# Iterate through year folders in retries
for year_folder in sorted(retries_dir.glob('*')):
    if year_folder.is_dir():
        # Iterate through JSON files in each year folder
        for json_file in year_folder.glob('*.json'):
            try:
                # Read JSON file
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                # Extract permit information
                permit_date = data.get('permitIssuanceDate', '')
                registration_number = data.get('registrationNumber', '')
                data_center_name = data.get('dataCenterName', '')
                
                # Process each equipment item
                for equipment in data.get('equipmentSummary', []):
                    # Get values, handling both schemas
                    num_units = equipment.get('numberOfUnits', 1.0) or 1.0
                    
                    # Handle electrical capacity - compute per unit if only total exists
                    elec_capacity_total = equipment.get('electricalCapacity_kW_total') or equipment.get('electricalCapacity_kW')
                    elec_capacity_per_unit = equipment.get('electricalCapacity_kW_perUnit')
                    
                    if elec_capacity_per_unit is None and elec_capacity_total is not None and num_units > 0:
                        elec_capacity_per_unit = elec_capacity_total / num_units
                    elif elec_capacity_per_unit is not None and elec_capacity_total is None:
                        elec_capacity_total = elec_capacity_per_unit * num_units
                    
                    # Handle mechanical capacity - compute per unit if only total exists
                    mech_capacity_total = equipment.get('mechanicalCapacity_bhp_total') or equipment.get('mechanicalCapacity_bhp')
                    mech_capacity_per_unit = equipment.get('mechanicalCapacity_bhp_perUnit')
                    
                    if mech_capacity_per_unit is None and mech_capacity_total is not None and num_units > 0:
                        mech_capacity_per_unit = mech_capacity_total / num_units
                    elif mech_capacity_per_unit is not None and mech_capacity_total is None:
                        mech_capacity_total = mech_capacity_per_unit * num_units
                    
                    # Handle gas usage
                    gas_usage = equipment.get('gasUsage_MMBTUhr_perUnit') or equipment.get('gasUsage_MMBTUhr')
                    
                    equipment_row = {
                        'Permit Date': permit_date,
                        'Registration Number': registration_number,
                        'Data Center Name': data_center_name,
                        'Type': equipment.get('type'),
                        'Reference No.': equipment.get('referenceNos'),
                        'Description': equipment.get('description'),
                        'Manufacturer (AI Inferred)': equipment.get('manufacturer'),
                        'Fuel Type (AI Inferred)': equipment.get('fuelType'),
                        'No. of Units': num_units,
                        'Electrical Capacity KW Per Unit': elec_capacity_per_unit,
                        'Total Electrical Capacity (AI Inferred)': elec_capacity_total,
                        'Mechanical Capacity bhp Per Unit': mech_capacity_per_unit,
                        'Total Mechanical Capacity (AI Inferred)': mech_capacity_total,
                        'Gas Usage': gas_usage,
                        'Controls (AI Inferred)': equipment.get('controls'),
                        'Add On Control Tech': equipment.get('addOnControlTechnology'),
                        'Original Permit Date (AI Inferred)': equipment.get('originalPermitDate')
                    }
                    retries_equipment.append(equipment_row)
            
            except Exception as e:
                print(f"Error processing {json_file}: {e}")

# Create DataFrame from retries data
df_retries = pd.DataFrame(retries_equipment)

# Append to original DataFrame
df_combined = pd.concat([df, df_retries], ignore_index=True)

print(f"\nOriginal DataFrame records: {len(df)}")
print(f"Retries DataFrame records: {len(df_retries)}")
print(f"Combined DataFrame records: {len(df_combined)}")
print(f"\nCombined DataFrame shape: {df_combined.shape}")

# Display sample from retries data
print("\nSample from retries data:")
df_retries.head()


Original DataFrame records: 819
Retries DataFrame records: 252
Combined DataFrame records: 1071

Combined DataFrame shape: (1071, 17)

Sample from retries data:


,Permit Date,Registration Number,Data Center Name,Type,Reference No.,Description,Manufacturer (AI Inferred),Fuel Type (AI Inferred),No. of Units,Electrical Capacity KW Per Unit,Total Electrical Capacity (AI Inferred),Mechanical Capacity bhp Per Unit,Total Mechanical Capacity (AI Inferred),Gas Usage,Controls (AI Inferred),Add On Control Tech,Original Permit Date (AI Inferred)
0,2013-08-27,73370,Ashburn Corporate Campus (ACC) data centers,Previously Permitted,"RPU-1, RPU-2, RPU-5, RPU-6, RPU-8","Five MTU Friedrichshafen, Rotary Uninterruptib...",None,Diesel,5.0,1800.0,9000.0,2937.0,14685.0,0.0,None,Steuler CERNOX-16V400/2000 SCR Open Loop System,2005-03-24
1,2013-08-27,73370,Ashburn Corporate Campus (ACC) data centers,Previously Permitted,"RPU-3, RPU-4, RPU-7","Three Detroit Diesel, Rotary UPS Engines, Mode...",None,Diesel,3.0,1800.0,5400.0,2935.0,8805.0,0.0,None,Steuler CERNOX-16V400/2000 SCR Open Loop System,2005-03-24
2,2013-08-27,73370,Ashburn Corporate Campus (ACC) data centers,Previously Permitted,"RPU-R1, RPU-R2","Two Detroit Diesel, Rotary UPS Engines, Model ...",None,Diesel,2.0,1800.0,3600.0,2935.0,5870.0,0.0,None,None,2005-03-24
3,2013-08-27,73370,Ashburn Corporate Campus (ACC) data centers,Previously Permitted,"EG-9, EG-10, EG-11, EG-12","Four Detroit Diesel, Stand-By Generators, Engi...",None,Diesel,4.0,1820.0,7280.0,2936.0,11744.0,0.0,None,None,2005-03-24
4,2013-08-27,73370,Ashburn Corporate Campus (ACC) data centers,Exempt,Tank 1 through Tank 14,Fourteen Above Ground Storage Tanks (AST) for ...,None,Diesel,14.0,0.0,0.0,0.0,0.0,0.0,None,None,2005-03-24


In [10]:
#sort dataframe by permit registration number and permit date
df_combined_sorted = df_combined.sort_values(by=['Registration Number', 'Permit Date'])
#drop manufacturer column
df_combined_sorted = df_combined_sorted.drop(columns=['Manufacturer (AI Inferred)'])

In [11]:
# Create aggregated dataframe - one row per data center (permit)
df_aggregated = df_combined.groupby(
    ['Permit Date', 'Registration Number', 'Data Center Name'], 
    dropna=False
).agg({
    'No. of Units': 'sum',
    'Total Electrical Capacity (AI Inferred)': 'sum',
    'Total Mechanical Capacity (AI Inferred)': 'sum',
    'Gas Usage': 'sum'
}).reset_index()

# Rename columns for clarity
df_aggregated = df_aggregated.rename(columns={
    'No. of Units': 'Total No. of Units',
    'Total Electrical Capacity (AI Inferred)': 'Total Electrical Capacity kW (AI Inferred)',
    'Total Mechanical Capacity (AI Inferred)': 'Total Mechanical Capacity bhp (AI Inferred)',
    'Gas Usage': 'Total Gas Usage MMBTUhr'
})

print(f"Aggregated DataFrame records: {len(df_aggregated)}")
print(f"Aggregated DataFrame shape: {df_aggregated.shape}")
print(f"\nColumn names:")
print(df_aggregated.columns.tolist())

# Display summary statistics
print(f"\nSummary Statistics:")
print(df_aggregated.describe())

# Display the aggregated data
df_aggregated.head(10)

Aggregated DataFrame records: 188
Aggregated DataFrame shape: (188, 7)

Column names:
['Permit Date', 'Registration Number', 'Data Center Name', 'Total No. of Units', 'Total Electrical Capacity kW (AI Inferred)', 'Total Mechanical Capacity bhp (AI Inferred)', 'Total Gas Usage MMBTUhr']

Summary Statistics:
       Total No. of Units  Total Electrical Capacity kW (AI Inferred)  \
count          188.000000                                1.880000e+02   
mean            61.409574                                1.443019e+05   
std             74.242537                                1.818059e+05   
min              1.000000                                0.000000e+00   
25%             12.000000                                1.520000e+04   
50%             39.000000                                7.800000e+04   
75%             88.250000                                2.237500e+05   
max            556.000000                                1.333650e+06   

       Total Mechanical Capacity b

,Permit Date,Registration Number,Data Center Name,Total No. of Units,Total Electrical Capacity kW (AI Inferred),Total Mechanical Capacity bhp (AI Inferred),Total Gas Usage MMBTUhr
0,2000-11-22,73170,"Exodus Communications, Incorporated (DC1)",2.0,2000.0,0.0,0.0
1,2003-09-16,73158,Qwest Communications Corporation,13.0,21000.0,0.0,0.0
2,2005-07-14,73326,Unisys Corporation,1.0,1250.0,1818.0,0.0
3,2006-10-19,73200,Verizon Business,10.0,14000.0,19936.0,0.0
4,2007-03-23,52173,"Corporate Office Properties, LP",3.0,7500.0,0.0,0.0
5,2007-07-26,73105,Freddie Mac corporate data center,4.0,8000.0,11392.0,0.0
6,2007-09-26,73369,"BP New Dominion Technology Park II, LLC",2.0,2000.0,2682.0,0.0
7,2007-10-18,73200,Verizon Business,7.0,0.0,0.0,0.0
8,2007-11-07,73567,"Digital Reston, LLC",2.0,4000.0,4380.0,0.0
9,2007-12-19,73643,"VISA U.S.A., Inc.",50.0,55000.0,84656.0,0.0


In [12]:
# save both dataframes to excel as separate sheets
with pd.ExcelWriter('power_deq_info_summary.xlsx') as writer:
    df_combined_sorted.to_excel(writer, sheet_name='Detailed Equipment Data', index=False)
    df_aggregated.to_excel(writer, sheet_name='Aggregated Permit Data', index=False)